In [ ]:
!pip install -q langchain langchain-openai transformers

In [ ]:
import os

from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, before_model, wrap_model_call, tool_call_limit
from langchain.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.runtime import Runtime
from pydantic import SecretStr
from transformers import TokenClassificationPipeline, pipeline
from typing import Callable, List


os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
api_key = SecretStr(userdata.get('OPENAI_API_KEY'))


def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
token_classifier = pipeline(task="token-classification", model="dbmdz/bert-large-cased-finetuned-conll03-english")

In [ ]:
@before_model
def append_best_regards_middleware(state: AgentState, runtime: Runtime) -> None:
    messages = state["messages"]
    last_message = messages[-1]
    print("The last message is: ", last_message)

    if isinstance(last_message, HumanMessage) and isinstance(last_message.content, str):
        print("Updating the messages...")

        # NOTE: It is important to use `model_copy` to preserve the message identifier.
        # Otherwise, the `add_messages` reducer used in `AgentState` will append instead of replace.
        return { "messages": last_message.model_copy(update={ "content": f"{last_message.content}\nBest Regards" }) }

# NOTE: To pass parameters to middlewares, we have a few options:
# - Higher-order function (just like the example below)
# - Constructor parameters (if we should work with classes)
# - Custom Agent State*
# - Config*
# - Context*
def make_entity_recognition_middleware(pipeline: TokenClassificationPipeline):
    @before_model
    def recognize_entities(state: AgentState, runtime: Runtime) -> None:
        messages = state["messages"]
        last_message = messages[-1]
        print("The last message is: ", last_message)

        if isinstance(last_message, HumanMessage) and isinstance(last_message.content, str):
            entities = pipeline(last_message.content, aggregation_strategy="simple")

            if entities:
                for entity in entities:
                    print(f"  - {entity['entity_group']} ({entity['score']:.4%}): \"{entity['word']}\"")
            else:
                print("No named entities were found.")

    return recognize_entities

@wrap_model_call
def use_dummy_responses(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
    print("Event: model_call")
    print(request)

    response = ModelResponse(result=[AIMessage("Hey! This is a dummy response. It costs you nothing!")])

    return response

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key),
    middleware=[
        make_entity_recognition_middleware(token_classifier),
        append_best_regards_middleware,
        use_dummy_responses,
        tool_call_limit
    ]
)

In [ ]:
prove_irrational = agent.invoke(
    input={
        "messages": [HumanMessage("Prove that the square root of 3 is irrational.")]
    }
)

In [ ]:
print_conversation(prove_irrational["messages"])

In [ ]:
red_hot_chilli_peppers_history = agent.invoke(
    input={
        "messages": [HumanMessage("Describe the top 3 most memorable concerts of Red Hot Chilli Peppers. Include information about the venue and whether or not Flea was naked.")]
    }
)

In [ ]:
political = agent.invoke(
    input={
        "messages": [HumanMessage("What decisions were made by Satya Nadella and Ursula von der Leyen on their recent meeting in Brussels?")]
    }
)

In [ ]:
print_conversation(political["messages"])